# Supply Chain Forecasting: Fully Interactive ML Dashboard
This notebook uses `ipywidgets` to create an interactive dashboard directly inside Jupyter. 
Instead of changing code, you can use the dropdown menu below to select **any product**. The Machine Learning models for **Delay Prediction** and **Demand Forecasting** will instantly re-train and update the graphs based on your selection.

In [1]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output

# 1. Load the core inventory data
inventory_data = pd.DataFrame({
    'Product_ID': ['P101', 'P102', 'P103', 'P104', 'P105', 'P106', 'P107', 'P108'],
    'Product_Name': ['Laptop', 'Mouse', 'Rice Bag', 'Charger', 'Milk Pack', 'Keyboard', 'Bread', 'Monitor'],
    'Warehouse': ['Bangalore', 'Chennai', 'Bangalore', 'Chennai', 'Bangalore', 'Bangalore', 'Chennai', 'Bangalore'],
    'Stock_Available': [15, 80, 25, 10, 150, 25, 60, 8],
    'Reorder_Level': [20, 50, 40, 30, 100, 20, 50, 15]
})

# 2. Define the main function that runs all ML and plotting
def analyze_product(target_product_id):
    # --- PROFILE SETUP ---
    product_info = inventory_data[inventory_data['Product_ID'] == target_product_id].iloc[0]
    product_name = product_info['Product_Name']
    warehouse = product_info['Warehouse']
    stock_available = product_info['Stock_Available']
    reorder_level = product_info['Reorder_Level']
    
    print(f"\n{'='*50}")
    print(f"📊 ACTIVE PROFILE: {product_name} ({target_product_id}) | Warehouse: {warehouse}")
    print(f"📦 Current Stock: {stock_available} | ⚠️ Reorder Level: {reorder_level}")
    print(f"{'='*50}\n")
    
    # Create a 1x2 grid for our two charts
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))
    
    # --- ML 1: SPHIMENT DELAY PREDICTION ---
    np.random.seed(int(target_product_id[-2:])) 
    delay_days_x = np.array([1,2,3,4,5,6,7,8]).reshape(-1,1)
    delay_days_y = np.random.randint(0, 4, size=8)
    
    delay_model = LinearRegression()
    delay_model.fit(delay_days_x, delay_days_y)
    
    delay_pred = delay_model.predict(delay_days_x)
    future_delay = delay_model.predict([[9]])[0]
    
    print(f"🚚 SHIPMENT PREDICITON:")
    print(f"   Expected delay for incoming shipment: {round(max(future_delay, 0), 2)} days")
    
    ax1.plot(delay_days_x, delay_days_y, marker='o', label='Actual Delay')
    ax1.plot(delay_days_x, delay_pred, marker='o', linestyle='--', label='Trend')
    ax1.set_title(f"Shipment Delay Prediction - {product_name}")
    ax1.set_xlabel("Shipment Number")
    ax1.set_ylabel("Delay (days)")
    ax1.legend()
    ax1.grid(True)

    # --- ML 2: DEMAND FORECASTING ---
    demand_days_x = np.array([1,2,3,4,5,6,7]).reshape(-1,1)
    base_demand = int(stock_available / 5) + 1
    orders_y = np.array([base_demand + i + np.random.randint(-2, 3) for i in range(7)])
    orders_y = np.maximum(orders_y, 0)
    
    demand_model = LinearRegression()
    demand_model.fit(demand_days_x, orders_y)
    
    future_demand = demand_model.predict([[8]])[0]
    predicted_orders = max(round(future_demand, 2), 0)
    pred_orders_y = demand_model.predict(demand_days_x)
    
    print(f"\n🛒 DEMAND FORECASTING:")
    if stock_available <= reorder_level:
        print(f"   ⚠️ ALERT: Already AT OR BELOW reorder capacity! ({stock_available} <= {reorder_level})")
    else:
        print(f"   ✅ STATUS: Stock is healthy.")
        
    print(f"   📈 Predicted orders for next day: {predicted_orders}")
    print(f"   📉 Projected stock balance tomorrow: {round(stock_available - predicted_orders, 2)}\n")

    ax2.plot(demand_days_x, orders_y, marker='o', label='Actual Orders')
    ax2.plot(demand_days_x, pred_orders_y, linestyle='--', color='orange', label='Forecast Trend')
    ax2.axhline(y=reorder_level, color='r', linestyle='-', alpha=0.3, label=f'Reorder Level ({reorder_level})')
    ax2.axhline(y=stock_available, color='g', linestyle='-', alpha=0.3, label=f'Current Stock ({stock_available})')
    ax2.set_title(f"Demand Forecast - {product_name} ({warehouse})")
    ax2.set_xlabel("Day")
    ax2.set_ylabel("Number of Orders")
    ax2.legend()
    ax2.grid(True)
    
    plt.show()

# 3. Create the interactive dropdown widget
# Map the display strings (e.g., "P101 - Laptop (Bangalore)") to the Product IDs
dropdown_options = {
    f"{row['Product_ID']} - {row['Product_Name']} ({row['Warehouse']})": row['Product_ID'] 
    for index, row in inventory_data.iterrows()
}

product_dropdown = widgets.Dropdown(
    options=dropdown_options,
    description='Select Product:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='400px')
)

# 4. Render the interactive UI
output_ui = widgets.interactive_output(analyze_product, target_product_id=product_dropdown)
display(product_dropdown, output_ui)

ModuleNotFoundError: No module named 'ipywidgets'